# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
# ===== CLEAN SETUP =====

!pip -q uninstall -y \
  langchain \
  langchain-core \
  langchain-classic \
  langchain-pinecone \
  pinecone-client \
  pinecone \
  >/dev/null 2>&1 || true

!pip -q install -U \
  numpy==1.26.4 \
  pyarrow==14.0.2 \
  datasets==2.18.0 \
  transformers \
  sentence-transformers \
  huggingface_hub \
  "pinecone[grpc]"

from google.colab import userdata
import os

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

print("SETUP_OK")

SETUP_OK


Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [2]:
# load the dataset from huggingface in streaming mode and shuffle it
from datasets import load_dataset
wiki_data = load_dataset(
    'vblagoje/wikipedia_snippets_streamed',
    split='train',
    streaming=True
).shuffle(seed=960)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for vblagoje/wikipedia_snippets_streamed contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/vblagoje/wikipedia_snippets_streamed
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the nex

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [3]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'wiki_id': 'Q7649565',
 'start_paragraph': 20,
 'start_character': 272,
 'end_paragraph': 24,
 'end_character': 380,
 'article_title': 'Sustainable Agriculture Research and Education',
 'section_title': "2000s & Evaluation of the program's effectiveness",
 'passage_text': "preserving the surrounding prairies. It ran until March 31, 2001.\nIn 2008, SARE celebrated its 20th anniversary. To that date, the program had funded 3,700 projects and was operating with an annual budget of approximately $19 million. Evaluation of the program's effectiveness As of 2008, 64% of farmers who had received SARE grants stated that they had been able to earn increased profits as a result of the funding they received and utilization of sustainable agriculture methods. Additionally, 79% of grantees said that they had experienced a significant improvement in soil quality though the environmentally friendly, sustainable methods that they were"}

In [54]:
# The 'wiki_snippets' dataset does not have 'section_title', so we will proceed without this specific filter
history = wiki_data.filter(lambda d: d['section_title'].startswith('History'))

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [55]:
from tqdm.auto import tqdm

total_doc_count = 50000

counter = 0
docs = []

for d in tqdm(history, total=total_doc_count):

    if d.get("section_title") and d["section_title"].startswith("History"):
        docs.append({
            "article_title": d.get("article_title"),
            "section_title": d.get("section_title"),
            "passage_text": d.get("passage_text")
        })
        counter += 1

    if counter >= total_doc_count:
        break

  0%|          | 0/50000 [00:00<?, ?it/s]

In [56]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,Taupo District,History,was not until the 1950s that the region starte...
1,Sutarfeni,History & Western asian analogues,Sutarfeni History strand-like pheni were Phena...
2,The Bishop Wand Church of England School,History,The Bishop Wand Church of England School Histo...
3,Teufelsmoor,History & Situation today,"made to preserve the original landscape, altho..."
4,Surface Hill Uniting Church,History,in perpetual reminder that work and worship go...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [7]:
from google.colab import userdata
import os
from pinecone import Pinecone, ServerlessSpec

pinecone_api_key = userdata.get('PINECONE_API_KEY')

spec = ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

pc = Pinecone(api_key=pinecone_api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [8]:
spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [9]:
index_name = "abstractive-question-answering"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )

index = pc.Index(index_name)
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 01 Mar 2026 09:07:10 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '48',
                                    'x-pinecone-request-latency-ms': '47',
                                    'x-pinecone-response-duration-ms': '49'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [10]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the retriever model from HuggingFace
retriever = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [61]:
from tqdm.auto import tqdm

total_doc_count = 50000
counter = 0
docs = []

for d in tqdm(history, total=total_doc_count):

    if d.get("section_title") and d["section_title"].startswith("History"):
        docs.append({
            "article_title": d.get("article_title"),
            "section_title": d.get("section_title"),
            "passage_text": d.get("passage_text")
        })
        counter += 1

    if counter >= total_doc_count:
        break

  0%|          | 0/50000 [00:00<?, ?it/s]

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [62]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [63]:
def format_query(query, context):

    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    context = " ".join(context)

    return f"question: {query} context: {context}"

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [29]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

QueryResponse(matches=[{'id': '49288',
 'metadata': {'article_title': 'Electric power system',
              'passage_text': 'Electric power system History In 1881, two '
                              "electricians built the world's first power "
                              'system at Godalming in England. It was powered '
                              'by two waterwheels and produced an alternating '
                              'current that in turn supplied seven Siemens arc '
                              'lamps at 250 volts and 34 incandescent lamps at '
                              '40 volts. However, supply to the lamps was '
                              'intermittent and in 1882 Thomas Edison and his '
                              'company, The Edison Electric Light Company, '
                              'developed the first steam-powered electric '
                              'power station on Pearl Street in New York City. '
                              'The Pearl 

In [30]:
from pprint import pprint

In [31]:
# format the query in the form generator expects the input
query = format_query(query, result['matches'])
pprint(query)

None


The output looks great. Now let's write a function to generate answers.

In [43]:
def generate_answer(query):

    if not isinstance(query, str):
        query = str(query)

    inputs = tokenizer(
        query,
        max_length=1024,
        return_tensors="pt",
        truncation=True
    ).to(device)

    ids = generator.generate(
        inputs["input_ids"],
        num_beams=2,
        min_length=20,
        max_length=40
    )

    answer = tokenizer.decode(ids[0], skip_special_tokens=True)
    return answer

In [66]:
generate_answer(query)

'The first wireless message was sent by Morse code. The first wireless message was sent by Morse code. The first wireless message was sent by radio. The first wireless message was sent by radio.'

As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [65]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

'The first wireless message was sent by Morse code. The first wireless message was sent by Morse code. The first wireless message was sent by radio. The first wireless message was sent by radio.'

To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [38]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

selects which antenna input is used; the first two bands use the low frequency antenna, and the last two bands select the high frequency antenna.
It is unknown whether the model 20B receiver had a beat frequency oscillator that would enable the detection of continuous wave transmissions such as Morse code and radiolocation beacons. Neither Earhart nor Noonan were capable of using Morse code. They relied on voice communications. Manning, who was on the first world flight attempt but not the second, was skilled at Morse and had acquired an FCC aircraft radiotelegraph license for 15 words per minute in March
---
YOU BY RADIO WE ARE FLYING AT A 1000 FEET
Earhart's 7:58 am transmission said she couldn't hear the Itasca and asked them to send voice signals so she could try to take a radio bearing. This transmission was reported by the Itasca as the loudest possible signal, indicating Earhart and Noonan were in the immediate area. They couldn't send voice at the frequency she asked for, so Mo

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [64]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

'COVID-19 is a retrovirus, which means that it is a virus that has been around for a long time. It was first discovered in 1993 in the Four Corners region of'

In [47]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

hands and arms. The virus causes a high fever and has the same intensity as that of an acute viral infection, similar to any virus present and visible in the bloodstream. What the GVN has done The GVN was initially formed in 2011 in response to the outbreak of the Chikungunya virus, when it had just spread to the Western Hemisphere. While the GVN also discussed tackling the ongoing Ebola crisis centered around West Africa, the Chikungunya virus was their main priority at the time. CHIKV was discovered a little before 1968, and outbreaks had taken place everywhere from Thailand
---
to the French Island of Réunion, where the virus at the time had caused 254 deaths. In 2013, the virus began to spread to the Caribbean and across the Atlantic to South America. The GVN is working toward antiviral drugs and vaccines against the Chikungunya virus, however one problem that the GVN has faced from the beginning is the limited ability to diagnose patients with the virus. As a result, the GVN was u

Let’s finish with a final few questions.

In [48]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

"I'm not a historian, but I'm a historian. I'm a historian. I'm a historian."

In [49]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

"I'm not a historian, but I'm a historian. I'm a historian. I'm a historian."

In [50]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

"I'm not a historian, but I'm a historian. I'm a historian. I'm a historian."

As we can see, the model can generate some decent answers.

#### Add a few more questions

In [51]:
query = "Who invented the electric light bulb?"
result = query_pinecone(query, top_k=1)
query = format_query(query, result['matches'])
generate_answer(query)

"I'm not a historian, but I'm a historian. I'm a historian. I'm a historian."

In [52]:
query = "When was the first railway system built?"
result = query_pinecone(query, top_k=1)
query = format_query(query, result['matches'])
generate_answer(query)

"I'm not a historian, but I'm a historian. I'm a historian. I'm a historian."